In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
simulation_summary = pd.read_csv("../../dataset/result/simulation/simulation_summary.csv")
simulation_details = pd.read_csv("../../dataset/result/simulation/simulation_details.csv")
simulation_summary_with_error = pd.read_csv("../../dataset/result/simulation/simulation_summary_with_forecast_error.csv")
simulation_details_with_error = pd.read_csv("../../dataset/result/simulation/simulation_details_with_forecast_error.csv")
simulation_summary_hybrid = pd.read_csv("../../dataset/result/simulation/simulation_summary_hybrid.csv")
simulation_details_hybrid = pd.read_csv("../../dataset/result/simulation/simulation_details_hybrid.csv")

simulation_summary['id'] = simulation_summary['id'].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)

In [37]:
simulation_summary_with_error['id']

0          FOODS_2_046_TX_1
1          FOODS_3_796_WI_2
2        HOBBIES_2_093_CA_2
3          FOODS_3_309_TX_2
4          FOODS_1_039_TX_1
                ...        
30485      FOODS_3_299_TX_1
30486      FOODS_3_455_WI_2
30487      FOODS_1_144_WI_1
30488    HOBBIES_2_055_WI_2
30489      FOODS_3_469_CA_1
Name: id, Length: 30490, dtype: str

In [49]:
simulation_summary.merge(
    simulation_summary_with_error[["id", "daily_service_level_empirical"]],
    on="id",
    how="left"
)[['id', 'daily_service_level_empirical_x', 'daily_service_level_empirical_y']].query("daily_service_level_empirical_x < 	daily_service_level_empirical_y")

,id,daily_service_level_empirical_x,daily_service_level_empirical_y
14,FOODS_1_002_TX_1,0.928571,0.964286
21,FOODS_1_003_CA_2,0.928571,0.964286
37,FOODS_1_004_WI_1,0.071429,0.107143
70,FOODS_1_009_CA_1,0.357143,0.464286
71,FOODS_1_009_CA_2,0.285714,0.357143
...,...,...,...
30398,HOUSEHOLD_2_507_WI_2,0.892857,0.964286
30411,HOUSEHOLD_2_509_CA_2,0.928571,1.000000
30413,HOUSEHOLD_2_509_CA_4,0.964286,1.000000
30436,HOUSEHOLD_2_511_TX_3,0.892857,0.964286


In [57]:
simulation_summary_with_error[simulation_summary_with_error["mu_error"] < 0]

,Unnamed: 0,review_period,lead_time,service_level_target,z_value,mu_error,std_error,safety_stock,initial_inventory,num_periods_simulated,...,item_id,dept_id,store_id,cat_id,state_id,unit_price,holding_cost_per_unit,shortage_cost_per_unit,estimation_window,simulation_window
0,0,28,0,0.95,1.644854,-0.075170,0.587737,5.115514,5.120792,28,...,1797,5,4,2,1,2.24,0.0224,0.5600,d_1884 to d_1913,d_1914 to d_1941
1,1,28,0,0.95,1.644854,-0.396969,0.384120,3.343280,0.000000,28,...,3017,6,8,2,2,2.98,0.0298,0.7450,d_1884 to d_1913,d_1914 to d_1941
3,3,28,0,0.95,1.644854,-0.180858,0.204490,1.779832,0.000000,28,...,2533,6,5,2,1,1.98,0.0198,0.4950,d_1884 to d_1913,d_1914 to d_1941
5,5,28,0,0.95,1.644854,-0.119699,0.526927,4.586241,3.895695,28,...,1443,3,7,1,2,8.97,0.0897,2.2425,d_1884 to d_1913,d_1914 to d_1941
6,6,28,0,0.95,1.644854,-0.287265,0.907615,7.899651,5.680999,28,...,975,2,2,1,0,6.42,0.0642,1.6050,d_1884 to d_1913,d_1914 to d_1941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30474,30474,28,0,0.95,1.644854,-0.318244,0.418446,3.642051,0.000000,28,...,2861,6,8,2,2,2.98,0.0298,0.7450,d_1884 to d_1913,d_1914 to d_1941
30476,30476,28,0,0.95,1.644854,-0.029420,0.717527,6.245171,8.604431,28,...,789,2,2,1,0,2.96,0.0296,0.7400,d_1884 to d_1913,d_1914 to d_1941
30477,30477,28,0,0.95,1.644854,-0.053920,0.967042,8.416890,11.147526,28,...,846,2,6,1,1,7.93,0.0793,1.9825,d_1884 to d_1913,d_1914 to d_1941
30483,30483,28,0,0.95,1.644854,-0.061091,0.558613,4.862023,4.669994,28,...,1861,5,4,2,1,2.50,0.0250,0.6250,d_1884 to d_1913,d_1914 to d_1941


In [6]:
group_cols = ["store_id", "cat_id"]

agg_dict = {
    # SUM
    "total_demand": "sum",
    "total_sales": "sum",
    "total_shortage_units": "sum",
    "total_order_qty": "sum",
    "total_holding_cost": "sum",
    "total_shortage_cost": "sum",
    "total_order_cost": "sum",
    "total_cost": "sum",
    "num_orders": "sum",

    # MEAN (rates)
    "fill_rate": "mean",
    "cycle_service_level_empirical": "mean",
    "daily_service_level_empirical": "mean",
    "average_ending_inventory": "mean",

    # MEAN (parameters)
    "mu_demand": "mean",
    "std_demand": "mean",
    "safety_stock": "mean",
    "order_up_to_level": "mean",
    "holding_cost_per_unit": "mean",
    "shortage_cost_per_unit": "mean"
}

summary_df = simulation_summary.agg(agg_dict).reset_index()
summary_df

,index,0
0,total_demand,1.231764e+06
1,total_sales,1.142095e+06
2,total_shortage_units,8.966856e+04
3,total_order_qty,8.625084e+05
4,total_holding_cost,3.031151e+05
5,total_shortage_cost,8.414164e+04
6,total_order_cost,3.979450e+04
7,total_cost,4.270512e+05
8,num_orders,7.958900e+04
9,fill_rate,9.126909e-01


In [4]:
agg_dict_2 = {
    # SUM
    "total_demand": "sum",
    "total_sales": "sum",
    "total_shortage_units": "sum",
    "total_order_qty": "sum",
    "total_holding_cost": "sum",
    "total_shortage_cost": "sum",
    "total_order_cost": "sum",
    "total_cost": "sum",
    "num_orders": "sum",

    # MEAN (rates)
    "fill_rate": "mean",
    "cycle_service_level_empirical": "mean",
    "daily_service_level_empirical": "mean",
    "average_ending_inventory": "mean",

    # MEAN (parameters)
    "safety_stock": "mean",
    "holding_cost_per_unit": "mean",
    "shortage_cost_per_unit": "mean"
}


summary_with_error_df = simulation_summary_with_error.agg(agg_dict_2).reset_index()
summary_with_error_df

,index,0
0,total_demand,1.231764e+06
1,total_sales,1.169462e+06
2,total_shortage_units,6.230177e+04
3,total_order_qty,8.891566e+05
4,total_holding_cost,3.268249e+05
5,total_shortage_cost,5.845107e+04
6,total_order_cost,4.616800e+04
7,total_cost,4.314440e+05
8,num_orders,9.233600e+04
9,fill_rate,9.542604e-01


In [10]:
simulation_summary_with_error.columns

Index(['Unnamed: 0', 'review_period', 'lead_time', 'service_level_target',
       'z_value', 'mu_error', 'std_error', 'safety_stock', 'initial_inventory',
       'num_periods_simulated', 'num_orders', 'total_demand', 'total_sales',
       'total_shortage_units', 'fill_rate', 'cycle_service_level_empirical',
       'daily_service_level_empirical', 'average_ending_inventory',
       'total_order_qty', 'total_holding_cost', 'total_shortage_cost',
       'total_order_cost', 'total_cost', 'id', 'item_id', 'dept_id',
       'store_id', 'cat_id', 'state_id', 'unit_price', 'holding_cost_per_unit',
       'shortage_cost_per_unit', 'estimation_window', 'simulation_window'],
      dtype='str')

In [16]:
group_cols = ["store_id", "cat_id"]

agg_dict = {
    # SUM
    "total_demand": "sum",
    "total_sales": "sum",
    "total_shortage_units": "sum",
    "total_order_qty": "sum",
    "total_holding_cost": "sum",
    "total_shortage_cost": "sum",
    "total_order_cost": "sum",
    "total_cost": "sum",
    "num_orders": "sum",

    # MEAN (rates)
    "fill_rate": "mean",
    "cycle_service_level_empirical": "mean",
    "daily_service_level_empirical": "mean",
    "average_ending_inventory": "mean",

    # MEAN (parameters)
    "mu_demand": "mean",
    "std_demand": "mean",
    "safety_stock": "mean",
    "order_up_to_level": "mean",
    "holding_cost_per_unit": "mean",
    "shortage_cost_per_unit": "mean"
}

summary_hybrid = simulation_summary_hybrid.agg(agg_dict).reset_index()
summary_hybrid

,index,0
0,total_demand,1.231764e+06
1,total_sales,1.056771e+06
2,total_shortage_units,1.749929e+05
3,total_order_qty,7.966685e+05
4,total_holding_cost,1.771613e+05
5,total_shortage_cost,1.626699e+05
6,total_order_cost,3.957200e+04
7,total_cost,3.794031e+05
8,num_orders,7.914400e+04
9,fill_rate,8.271364e-01
